In [ ]:
# Operate from the parent directory
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import torch
from utils.animation.processing.bvh_converter import BVHParser
import utils.animation.visualisation.old_visualisation as animation_visualisation
import time


In [ ]:
# Initialize visualization
animation_visualisation.init_visualization()

In [ ]:
target_joints = ['body_world', 'b_root', 'b_r_foot', 'b_l_foot', 'b_l_upleg', 'b_l_leg', 'b_r_upleg', 'b_r_leg', 
                'b_spine0', 'b_spine1', 'b_spine2', 'b_spine3', 'b_l_shoulder', 'b_l_arm', 'b_l_arm_twist', 
                'b_l_forearm', 'b_l_wrist_twist', 'b_l_wrist', 'b_l_pinky1', 'b_l_pinky2', 'b_l_pinky3', 'b_l_ring1', 
                'b_l_ring2', 'b_l_ring3', 'b_l_middle1', 'b_l_middle2', 'b_l_middle3', 'b_l_index1', 'b_l_index2', 
                'b_l_index3', 'b_l_thumb0', 'b_l_thumb1', 'b_l_thumb2', 'b_l_thumb3', 'b_r_shoulder', 'b_r_arm', 
                'b_r_arm_twist', 'b_r_forearm', 'b_r_wrist_twist', 'b_r_wrist', 'b_r_thumb0', 'b_r_thumb1', 
                'b_r_thumb2', 'b_r_thumb3', 'b_r_pinky1', 'b_r_pinky2', 'b_r_pinky3', 'b_r_middle1', 'b_r_middle2', 
                'b_r_middle3', 'b_r_ring1', 'b_r_ring2', 'b_r_ring3', 'b_r_index1', 'b_r_index2', 'b_r_index3', 
                'b_neck0', 'b_head']

bvh_file = "dataset/genea2023_dataset/toy/main-agent/bvh/trn_2023_v0_000_main-agent.bvh"

# Load the BVH file and extract features
parser = BVHParser(bvh_file, target_joints)

features = torch.tensor(parser.to_features()).to(torch.float32)

# Calculate frames to stream
num_frames = features.shape[0]

# I want to calculate the world position of the joints.
world_positions = parser.skeleton.calculate_world_positions(frame_data=features)

# reshape to (batch_size, num_frames, num_joints, 3)
world_positions = world_positions.reshape(num_frames, len(parser.skeleton.target_joints), 3)

# Create frames
for i in range(num_frames):
    frame_message = {
        "frameIndex": i,
        "totalFrames": num_frames
    }
    frame_message = animation_visualisation.add_pose_to_message(features[i], parser.skeleton, frame_message)
    # frame_message = animation_visualisation.add_debug_positions_to_message(world_positions[i],frame_message)
    frame_message = animation_visualisation.add_debug_text_to_message(f"Frame {i+1}/{num_frames}", frame_message)
    
    animation_visualisation.send_message(frame_message)

    time.sleep(1.0/30.0)